<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2008%20-%202026-05-04%20-%20Estadistica%20descriptiva%20y%20visualizacion/clase_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 8 · Estadística descriptiva

Antes de que cualquier ML tenga sentido, tienes que saber describir lo que tienes. Media, mediana, distribución — básico pero ignorado. Hoy aprendes a calcular e interpretar. Y a ver cuando algo está "roto" (outliers).

In [ ]:
import random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
print("Setup completo")

## Comprendiendo distribuciones

¿Cómo se distribuyen las edades en el Titanic? ¿Hay outliers? Primero los números, luego los gráficos:

In [ ]:
titanic = sns.load_dataset("titanic").dropna(subset=["age"])

stats = titanic["age"].describe()
print("Estadísticas básicas de edad:")
print(stats.round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(titanic["age"], bins=30, edgecolor="black", alpha=0.7, color="skyblue")
axes[0].set_title("Histograma de edad")
axes[0].set_xlabel("Edad")

axes[1].boxplot(titanic["age"])
axes[1].set_title("Boxplot de edad")
axes[1].set_ylabel("Edad")

axes[2].hist(titanic["age"], bins=30, cumulative=True, edgecolor="black", alpha=0.7, color="coral")
axes[2].set_title("CDF de edad")
axes[2].set_xlabel("Edad")

plt.tight_layout()
plt.show()

`.describe()` calcula media, std, percentiles automáticamente. Las gráficas lo visualizan. Histograma muestra forma. Boxplot muestra mediana y quartiles. CDF muestra acumulativo.

## Estadísticas y percentiles

Calcula media, mediana, Q1, Q3 para la tarifa (fare):

In [ ]:
# Completa los huecos
media = titanic["fare"].______()
mediana = titanic["fare"].______()
q1 = titanic["fare"]._________(0.25)
q3 = titanic["fare"]._________(0.75)
iqr = q3 - q1

print(f"Media: ${media:.2f}")
print(f"Mediana: ${mediana:.2f}")
print(f"Q1: ${q1:.2f}, Q3: ${q3:.2f}")
print(f"IQR: ${iqr:.2f}")

In [ ]:
assert media > 0, "Media debe ser positiva"
assert mediana < media, "Típicamente la mediana es menor que la media (distribución derecha)"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# media = titanic["fare"].mean()
# mediana = titanic["fare"].median()
# q1 = titanic["fare"].quantile(0.25)
# q3 = titanic["fare"].quantile(0.75)

¿Por qué la mediana es menor que la media? Hay algunos pasajeros muy ricos que pagaron tarifas enormes. Eso sesga el promedio. La mediana no se deja sesgar — es el valor del medio.

## Detección de outliers con IQR

Encuentra outliers en "fare" usando la regla IQR: < Q1-1.5*IQR o > Q3+1.5*IQR. Contexto: limpieza de datos:

In [ ]:
# Calcula límites y filtra
q1 = titanic["fare"].quantile(0.25)
q3 = titanic["fare"].quantile(0.75)
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

# Indexación booleana para outliers
outliers = titanic[(titanic["fare"] < limite_inferior) | (titanic["fare"] > limite_superior)]

print(f"Límites: [{limite_inferior:.2f}, {limite_superior:.2f}]")
print(f"Outliers encontrados: {len(outliers)} ({len(outliers)/len(titanic)*100:.1f}%)")
print(f"Rango de outliers: [{outliers['fare'].min():.2f}, {outliers['fare'].max():.2f}]")

In [ ]:
assert len(outliers) > 0, "Deberías encontrar al menos 1 outlier"
assert (outliers["fare"] > limite_superior).any() or (outliers["fare"] < limite_inferior).any()
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# Ya está hecho arriba

¿Son todos estos outliers realmente "malos"? ¿Deberías removerlos? Aquí viene la honestidad técnica: depende. Si alguien pagó USD 512 por un pasaje de primera clase, eso es raro pero real. No es un error de medición. No lo elimines sin pensar.

## Dashboard exploratorio

Sin skeleton. Quieres un resumen visual rápido de edad, tarifa y supervivencia. Pasos: 4 subplots — histograma, boxplot, scatterplot, barplot.

In [ ]:
# Tu código aquí
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Subplot 1: histograma edad
# axes[0, 0].hist(titanic["age"], bins=20, edgecolor="black", alpha=0.7, color="blue")
# axes[0, 0].set_title("Distribución de edad")

# Subplot 2: boxplot tarifa
# axes[0, 1].boxplot(titanic["fare"])
# axes[0, 1].set_title("Tarifa (boxplot)")

# Subplot 3: scatterplot edad vs tarifa
# sns.scatterplot(data=titanic, x="age", y="fare", hue="survived", ax=axes[1, 0], alpha=0.6)
# axes[1, 0].set_title("Edad vs Tarifa (coloreado por supervivencia)")

# Subplot 4: barplot supervivencia por sexo
# sns.barplot(data=titanic, x="sex", y="survived", ax=axes[1, 1], ci=95)
# axes[1, 1].set_title("Tasa supervivencia por sexo")
# axes[1, 1].set_ylabel("Tasa de supervivencia")

plt.tight_layout()
plt.show()

print("Dashboard completado")

In [ ]:
# Criterios de aceptación
assert "fig" in dir() and "axes" in dir(), "Falta figura"
print("✓ Ejercicio completado")

In [ ]:
# %% [solution]
# Ya está en el código anterior (comentado)

## Relación entre variables

¿Hay relación entre edad y tarifa? Calculamos covarianza y correlación:

In [ ]:
# Covarianza entre edad y tarifa
cov = titanic["age"].cov(titanic["fare"])
corr = titanic["age"].corr(titanic["fare"])

print(f"Covarianza: {cov:.2f}")
print(f"Correlación de Pearson: {corr:.3f}")

# ¿Hay relación? Débil positiva.
print(f"\nInterpretación: {"""débil""" if abs(corr) < 0.3 else """moderada""" if abs(corr) < 0.7 else """fuerte"""} relación positiva")

Débil relación positiva: pasajeros más viejos pagaban un poco más, pero la edad no es el factor principal en la tarifa. La clase de pasaje importa más.

## Checkpoint

1. **Media vs. Mediana:** usa la media si datos normales, mediana si hay outliers.
2. **IQR:** intervalo entre Q1 y Q3, donde vive el 50% central de datos.
3. **Outlier IQR:** valor < Q1-1.5*IQR o > Q3+1.5*IQR.

Si algo no quedó claro, ahora es el momento de preguntar.

In [ ]:
assert "media" in dir() and "mediana" in dir(), "Falta variable"
print("Checkpoint ✓ — puedes continuar")

## El flujo de la exploración

```
Datos crudos
      ↓
  .describe() → media, std, percentiles
      ↓
  Histogramas + Boxplots → forma de distribución
      ↓
  Detectar outliers (IQR, Z-score)
      ↓
  Scatterplots → relaciones bivariadas
      ↓
  Decisiones de limpieza y transformación
```

## Para recordar

- **`.describe()`** para estadísticas rápidas: media, std, percentiles.
- **IQR (Rango intercuartílico):** Q3 - Q1, usado para detectar outliers.
- **Outliers:** típicamente valores > Q3 + 1.5*IQR o < Q1 - 1.5*IQR.
- **Histogramas + Boxplots** para visualizar distribuciones.
- **Scatterplots** para ver relaciones entre dos variables continuas.
- **Correlación de Pearson** para cuantificar relación lineal (-1 a 1).

## Mañana

Veremos **SQL con Python** — consultar bases de datos, joins, agregaciones. Es el cierre de la Semana 2 y la primera entrega del proyecto.

Para preparar:
- Relee distribuciones en el HTML si algo quedó confuso.
- Practica con otro dataset: `sns.load_dataset("iris")` o `sns.load_dataset("tips")`.

## Referencias

- [Pandas Descriptive Statistics](https://pandas.pydata.org/docs/user_guide/basics.html#descriptive-statistics)
- [Understanding Box Plots](https://towardsdatascience.com/understanding-boxplots-5e2df7bcbd51)
- [EDA with Seaborn](https://seaborn.pydata.org/examples/index.html)

## Reflexión

En tus propias palabras, explícale a un compañero **por qué la mediana es mejor que la media cuando hay outliers** en máximo 3 oraciones.

---

## Para tu equipo

**Análisis univariado + bivariado:**

- Para 3 variables numéricas claves:
  - Calcula describe(), percentiles
  - Detecta outliers (IQR)
  - Crea histogramas + boxplots
- Análisis bivariado:
  - Calcula correlaciones
  - Crea 2 scatterplots de variables importantes
- Documenta hallazgos

**Entregable:** `proyecto_analisis_estadistico.ipynb` con dashboard exploratorio.